## 分组

### 1.分组模式及其对象

#### 1.1 分组的一般模式

##### 练一练：请在learn_pandas数据集上按学校分组统计体重的均值。

In [37]:
import pandas as pd
import numpy as np
from sqlalchemy.util import ellipses_string

df = pd.read_csv("data_base/data/learn_pandas.csv")
df.groupby('School')['Weight'].mean()

School
A    56.442308
B    55.666667
C    54.000000
D    54.223881
Name: Weight, dtype: float64

#### 1.2 分组依据的本质

In [7]:
q25 = df.Weight.quantile(0.25)
q75 = df.Weight.quantile(0.75)
w_dict = {0:"low", 1:"normal", 2:"high"}
condition = ((df.Weight > q25)*1 + (df.Weight > q75)*1).replace(w_dict)
df.groupby(condition)["Height"].mean()

Weight
high      174.935714
low       155.891071
normal    162.255294
Name: Height, dtype: float64

#### 1.3 groupby对象

In [9]:
gp = df.groupby(['School','Grade'])
gp

In [10]:
# 分组的数量
gp.ngroups

16

In [13]:
res = gp.groups
res.keys()

dict_keys([('A', 'Freshman'), ('A', 'Junior'), ('A', 'Senior'), ('A', 'Sophomore'), ('B', 'Freshman'), ('B', 'Junior'), ('B', 'Senior'), ('B', 'Sophomore'), ('C', 'Freshman'), ('C', 'Junior'), ('C', 'Senior'), ('C', 'Sophomore'), ('D', 'Freshman'), ('D', 'Junior'), ('D', 'Senior'), ('D', 'Sophomore')])

In [15]:
# size()
df.iloc[:5,:5].size
gp.size()

School  Grade    
A       Freshman     13
        Junior       17
        Senior       22
        Sophomore     5
B       Freshman     13
        Junior        8
        Senior        8
        Sophomore     5
C       Freshman      9
        Junior       12
        Senior       11
        Sophomore     8
D       Freshman     17
        Junior       22
        Senior       14
        Sophomore    16
dtype: int64

In [17]:
gp.get_group(('A','Freshman')).iloc[:5,:5]

,School,Grade,Name,Gender,Height
0,A,Freshman,Gaopeng Yang,Female,158.9
6,A,Freshman,Qiang Chu,Female,162.5
10,A,Freshman,Xiaopeng Zhou,Male,174.1
60,A,Freshman,Yanpeng Lv,Male,NaN
114,A,Freshman,Xiaopeng Zhao,Female,161.0


### 2.聚合函数 -> 本质是对Series类型进行聚合

#### 2.1内置聚合函数

In [29]:
# 练一练:在learn_pandas数据集中，Transfer列的元素为“N”时表示该名同学不是转系生，请按照学校和年级两列分组，找出所有不含转系生的组对应的学校和年级。

res = (df.Transfer=="N").groupby([df.School, df.Grade]).all()
res # 等价于res[res == True]

School  Grade    
A       Freshman      True
        Junior        True
        Senior       False
        Sophomore     True
B       Freshman     False
        Junior       False
        Senior       False
        Sophomore    False
C       Freshman      True
        Junior       False
        Senior       False
        Sophomore     True
D       Freshman     False
        Junior       False
        Senior       False
        Sophomore    False
Name: Transfer, dtype: bool

In [8]:
s = pd.Series([True,False,True,False])
s[s]

0    True
2    True
dtype: bool

#### 2.2 agg()函数

In [10]:
# 练一练：请使用传入字典的方法完成与gb.agg(['max', 'min'])等价的聚合任务。

gb = df.groupby('Gender')[['Weight','Height']]
gb.agg({'Weight':['max','min'],'Height':['max','min']})

Weight       Height       
          max   min    max    min
Gender                           
Female   63.0  34.0  170.2  145.4
Male     89.0  51.0  193.9  155.7

In [12]:
gb.describe()

Weight                                                     Height  \
        count       mean       std   min   25%   50%    75%   max  count   
Gender                                                                     
Female  135.0  47.918519  5.405983  34.0  44.0  48.0  52.00  63.0  132.0   
Male     54.0  72.759259  7.772557  51.0  69.0  73.0  78.75  89.0   51.0   

                                                                    
             mean       std    min      25%    50%      75%    max  
Gender                                                              
Female  159.19697  5.053982  145.4  155.675  159.6  162.825  170.2  
Male    173.62549  7.048485  155.7  168.900  173.4  177.150  193.9

In [16]:
# 练一练：在groupby对象上可以使用describe()方法进行统计信息汇总，请同时使用多个聚合函数，完成与该方法相同的功能。

gb.agg(
    ['count','mean','std','min',
     ('25%',lambda x:x.quantile(0.25)), # 聚合结果重命名
     ('50%',lambda x:x.quantile(0.5)),
     ('75%',lambda x:x.quantile(0.75)),
     'max']
)

Weight                                                     Height  \
        count       mean       std   min   25%   50%    75%   max  count   
Gender                                                                     
Female    135  47.918519  5.405983  34.0  44.0  48.0  52.00  63.0    132   
Male       54  72.759259  7.772557  51.0  69.0  73.0  78.75  89.0     51   

                                                                    
             mean       std    min      25%    50%      75%    max  
Gender                                                              
Female  159.19697  5.053982  145.4  155.675  159.6  162.825  170.2  
Male    173.62549  7.048485  155.7  168.900  173.4  177.150  193.9

In [20]:
def my_func(s):
    res = 'High'
    if s.mean() < df[s.name].mean():
        res = 'Low'
    return res
gb.agg(my_func)

,Weight,Height
Gender,,
Female,Low,Low
Male,High,High


### 3.变换和过滤

#### 1.变换函数

In [3]:
import pandas as pd

example = pd.DataFrame({"A":list('abbaab'),"B":[3,5,6,2,1,7]})
example.groupby("A")["B"].cummax() # 先分组计算，后排序。

0    3
1    5
2    6
3    3
4    3
5    7
Name: B, dtype: int64

In [3]:
#  练一练：transform()方法无法像agg()一样，通过传入字典来对指定列使用特定的变换，如果需要在一次transform()的调用中实现这种功能，请给出解决方案。

import pandas as pd

def helper(x):
    """ 通过判断组名进行特定的运算 """
    if x.name == 'A':
        return x + 1
    elif x.name == 'B':
        return x - 1

df = pd.DataFrame({'A':[4,5,6,7],'B':[1,2,3,4],'C':list('aabb')})
df.groupby('C').transform(helper)

,A,B
0,5,0
1,6,1
2,7,2
3,8,3


In [8]:
# 练一练：在groupby对象中，rank()方法也是一个实用的变换函数，请在官方文档中查阅它的功能并给出1个使用的例子。

df = pd.read_csv("data_base/data/learn_pandas.csv")
df["年级内体重排名"] = df.groupby("Grade")["Weight"].rank(ascending=False)
# 体重最重排第一

df.loc[df["年级内体重排名"] < 3,["Grade","Name","Weight","年级内体重排名"]].sort_values(["Grade","年级内体重排名"])

,Grade,Name,Weight,年级内体重排名
38,Freshman,Qiang Han,87.0,1.0
99,Freshman,Changpeng Zhao,83.0,2.5
117,Freshman,Chunli Zhao,83.0,2.5
82,Junior,Changfeng Lv,76.0,1.5
158,Junior,Chengqiang Zhang,76.0,1.5
2,Senior,Mei Sun,89.0,1.0
23,Senior,Qiang Zheng,87.0,2.0
71,Sophomore,Feng Han,82.0,1.0
40,Sophomore,Li Wang,79.0,2.5
48,Sophomore,Mei Xu,79.0,2.5


In [11]:
gb.transform('mean').head()

,Weight,Height
0,47.918519,159.19697
1,72.759259,173.62549
2,72.759259,173.62549
3,47.918519,159.19697
4,72.759259,173.62549


In [12]:
example.groupby("A")["A"].transform(lambda x:0 if x.name == "a"else 1)

0    0
1    1
2    1
3    0
4    0
5    1
Name: A, dtype: int64

#### 2.组索引和过滤

In [13]:
gb.filter(lambda x:x.shape[0] > 100).head()

,Weight,Height
0,46.0,158.9
3,41.0,NaN
5,51.0,158.0
6,52.0,162.5
7,50.0,161.9


In [26]:
#练习：找出所有"没有转系生"的 (School, Grade) 组。(不方便)
res = df.groupby(['School', 'Grade']).filter(
    lambda x: (x['Transfer'] == 'N').all()
)
res.loc[:,['School', 'Grade']].drop_duplicates().sort_values(['School','Grade'])

,School,Grade
0,A,Freshman
31,A,Junior
13,A,Sophomore
15,C,Freshman
3,C,Sophomore


In [30]:
# 练一练：从概念上说，索引功能是组过滤功能的子集，请使用groupby对象上的filter()方法完成loc[...]的功能，这里假设“...”是元素的列表。
df_new = pd.DataFrame({"A": [1, 2, 3, 4, 5, 6]},index=list('abcdef'))

,A
a,1
b,2
c,3
d,4
e,5
f,6


In [31]:
item_list = ["b","d","f"]
df_new.groupby(df.index).filter(lambda x:x.index[0] in item_list)

,A
b,2
d,4
f,6


### 4.跨列分组

#### 1.标量的情况

In [46]:
gb1 = df.groupby(['Gender','Test_Number'])['Height']
gb2 = df.groupby(['Gender','Test_Number'])[['Height','Weight']]
gb1.apply(lambda x: 0)

Gender  Test_Number
Female  1              0
        2              0
        3              0
Male    1              0
        2              0
        3              0
Name: Height, dtype: int64

In [50]:
gb2.apply(lambda x: [0,0])

Gender  Test_Number
Female  1              [0, 0]
        2              [0, 0]
        3              [0, 0]
Male    1              [0, 0]
        2              [0, 0]
        3              [0, 0]
dtype: object

#### 2.Series的情况

In [51]:
gb1.apply(lambda x: pd.Series([0,1],index=['a','b']))

Gender  Test_Number   
Female  1            a    0
                     b    1
        2            a    0
                     b    1
        3            a    0
                     b    1
Male    1            a    0
                     b    1
        2            a    0
                     b    1
        3            a    0
                     b    1
Name: Height, dtype: int64

In [52]:
gb2.apply(lambda x: pd.Series([0,1],index=['a','b']))

a  b
Gender Test_Number      
Female 1            0  1
       2            0  1
       3            0  1
Male   1            0  1
       2            0  1
       3            0  1

#### 3.DataFrame

In [55]:
import numpy as np

temp_df = pd.DataFrame(np.ones((2,2)),
                       index=['a','b'],
                       columns=pd.Index([('w','x'),('y','z')]))
gb1.apply(lambda x:temp_df).head()

w    y
                        x    z
Gender Test_Number            
Female 1           a  1.0  1.0
                   b  1.0  1.0
       2           a  1.0  1.0
                   b  1.0  1.0
       3           a  1.0  1.0

In [56]:
gb2.apply(lambda x:temp_df).head()

w    y
                        x    z
Gender Test_Number            
Female 1           a  1.0  1.0
                   b  1.0  1.0
       2           a  1.0  1.0
                   b  1.0  1.0
       3           a  1.0  1.0

In [6]:
# 练一练：在groupby对象中还定义了cov()和corr()函数，从概念上说也属于跨列的分组处理。请利用本节定义的gb对象，使用apply()函数实现与gb.cov()同样的功能。
import numpy as np

df = pd.DataFrame(np.random.rand(12, 5),columns=list('ABCDE'))
df['F'] = list('aaaabbbbcccc')

In [9]:
apply_method = df.groupby('F').apply(lambda x: x.drop(columns='F').cov())
inner_method = df.groupby('F').cov()
apply_method.equals(inner_method)

/var/folders/qz/sxl6t5150772tp8x0p4g44mh0000gn/T/ipykernel_26639/247490668.py:1: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  apply_method = df.groupby('F').apply(lambda x: x.drop(columns='F').cov())


True

### 5.章末习题

#### 一、汽车数据的分组分析

In [11]:
# 第一题：1.筛选出所属Country数超过2个的汽车，即若该汽车的Country在总体数据集中出现次数不超过2则剔除。
df = pd.read_csv("data_base/data/ch4/car.csv")
df.head()

,Brand,Price,Country,Reliability,Mileage,Type,Weight,Disp.,HP
0,Eagle Summit 4,8895,USA,4.0,33,Small,2560,97,113
1,Ford Escort 4,7402,USA,2.0,33,Small,2345,114,90
2,Ford Festiva 4,6319,Korea,4.0,37,Small,1845,81,63
3,Honda Civic 4,6635,Japan/USA,5.0,32,Small,2260,91,92
4,Mazda Protege 4,6599,Japan,5.0,32,Small,2440,113,103


In [20]:
df[df.groupby('Country')['Country'].transform('size') > 2]

Country
USA          26
Japan        19
Japan/USA     7
Korea         3
Name: count, dtype: int64

In [23]:
# 第一题：2.按Country分组计算价格均值、价格变异系数以及该Country的汽车数量，其中变异系数的计算方法是标准差除以均值，并在结果中把变异系数重命名为CoV。

df.groupby('Country')['Price'].agg([
    ('价格均值','mean'),
    ('Cov',lambda x:x.std()/x.mean()),
    ('汽车数量','count')
])

,价格均值,Cov,汽车数量
Country,,,
France,15930.000000,NaN,1
Germany,14447.500000,0.435839,2
Japan,13938.052632,0.387429,19
Japan/USA,10067.571429,0.240040,7
Korea,7857.333333,0.243435,3
Mexico,8672.000000,NaN,1
Sweden,18450.000000,NaN,1
USA,12543.269231,0.203344,26


In [24]:
# 第二题：按照表中位置的前三分之一、中间三分之一和后三分之一分组，统计Price的均值。

k = len(df) // 3
label = np.repeat(['前1/3','中1/3','后1/3'],k)
df.groupby(label)['Price'].mean()

中1/3    13356.40
前1/3     9069.95
后1/3    15420.65
Name: Price, dtype: float64

In [29]:
# 第三题：
# 1.对Price和HP分别计算最大值和最小值，结果会产生多级列索引，请用下划线连接的方式把多级列索引合并为单层索引。

gb = df.groupby('Type')
res = gb.agg({'Price':['max'],'HP':['min']})
res.columns = res.columns.map(lambda x:'_'.join(x))
res

,Price_max,HP_min
Type,,
Compact,18900,95
Large,17257,150
Medium,24760,110
Small,9995,63
Sporty,13945,92
Van,15395,106


In [31]:
# 第三题:
# 2.对HP进行组内的min-max归一化，即每个元素减去组内HP的最小值后，再除以组内HP的极差。

gb['HP'].transform(lambda x:(x-x.min()) / (x.max()-x.min())).head()

0    1.00
1    0.54
2    0.00
3    0.58
4    0.80
Name: HP, dtype: float64

#### 二、某海洋物种在三大海域的分布研究

In [47]:
# 第一题：分组计算各年份在各海域的观测次数与海水盐度均值。
import numpy as np

df = pd.read_csv("data_base/data/ch4/marine_observation.csv")
Pacific = (df.longitude > -160) & (df.longitude < -120) & (df.latitude > -40) & (df.latitude < 0)
Indian = (df.longitude > 60) & (df.longitude < 100) & (df.latitude > -40) & (df.latitude < 0)
Atlantic = (df.longitude > -40) & (df.longitude < 0) & (df.latitude > -60) & (df.latitude < -20)

area = np.select([Pacific,Indian,Atlantic],['Pacific','Indian','Atlantic'],default='Atlantic')
year = df.date.str[:4].astype("int64")
df.groupby([year,area])['salinity'].agg(['size','mean'])

size       mean
date                           
2001 Atlantic  20073  35.015785
     Indian    19972  35.009664
     Pacific   19880  34.988308
2002 Atlantic  20230  34.988722
     Indian    20036  35.015698
     Pacific   19945  34.996261
2003 Atlantic  19746  35.002013
     Indian    19852  35.010008
     Pacific   19982  35.030261
2004 Atlantic  19816  35.035913
     Indian    19867  35.019048
     Pacific   20279  35.042076
2005 Atlantic  20216  35.009535
     Indian    20067  35.012394
     Pacific   20174  35.044264
2006 Atlantic  19868  34.959015
     Indian    19937  34.974679
     Pacific   19778  35.025384
2007 Atlantic  20165  34.986008
     Indian    20176  35.033520
     Pacific   19932  34.993681
2008 Atlantic  19984  34.984599
     Indian    20080  35.038688
     Pacific   19958  35.041478
2009 Atlantic  19762  34.994788
     Indian    20237  34.988864
     Pacific   19921  35.014393
2010 Atlantic  19893  34.985960
     Indian    19964  34.982203
     Pacific   20103  34.998150
2011 Atlantic  20016  34.978894
     Indian    20189  34.998509
     Pacific   19709  34.982122
2012 Atlantic  20044  35.017487
     Indian    20123  34.991866
     Pacific   20344  35.005662
2013 Atlantic  20136  35.007385
     Indian    19940  35.001068
     Pacific   19826  35.005815
2014 Atlantic  19923  34.983745
     Indian    19805  34.995801
     Pacific   19823  35.002196
2015 Atlantic  20011  35.000099
     Indian    20121  35.015773
     Pacific   19896  35.048375
2016 Atlantic  19984  35.015707
     Indian    20121  35.001149
     Pacific   20065  34.997276
2017 Atlantic  20024  34.998852
     Indian    19914  34.993537
     Pacific   20043  35.026888
2018 Atlantic  20093  35.012976
     Indian    19822  34.997731
     Pacific   20133  34.995871
2019 Atlantic  19921  34.992067
     Indian    19639  34.998293
     Pacific   19857  35.027713
2020 Atlantic  20095  35.001130
     Indian    20138  35.037383
     Pacific   20352  34.983448